# Training Notebook (Colab) — self-contained dev version

All pipeline code (data splitting, dataset, model, Lightning module) is inlined directly
into this notebook's cells for now, so you can edit and re-run without committing/pushing
to GitHub each time. Once the approach is stable, copy the cells back into
`data_splitter.py`, `dataset.py`, `model.py`, and `pytorch_lightning.py` and commit.

**Before running:** add a Colab secret (key icon in the left sidebar) named `KAGGLE_API_TOKEN`
with your Kaggle API token as the value (kaggle.com/settings → API). Toggle notebook access on.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('/content/beilinson')
(PROJECT_ROOT / 'data').mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / 'artifacts').mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)

In [ ]:
%pip install -q lightning kagglehub

In [ ]:
import os

from google.colab import userdata
import kagglehub

os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

data_dir = PROJECT_ROOT / 'data'

if not any(data_dir.glob('*/*')):
    print('Downloading dataset from Kaggle...')
    cache_path = Path(kagglehub.dataset_download('hasyimabdillah/workoutexercises-images'))

    # kagglehub caches the dataset under its own path; if it's wrapped in one extra
    # top-level folder, look inside that instead of the cache root.
    source = cache_path
    entries = list(source.iterdir())
    if len(entries) == 1 and entries[0].is_dir():
        source = entries[0]

    for class_dir in source.iterdir():
        if class_dir.is_dir():
            target = data_dir / class_dir.name
            if not target.exists():
                target.symlink_to(class_dir, target_is_directory=True)
else:
    print('Dataset already present, skipping download.')

class_dirs = sorted(p.name for p in data_dir.iterdir() if p.is_dir())
print(f'{len(class_dirs)} classes found:', class_dirs)

## Inlined modules

The cells below are the contents of `data_splitter.py`, `dataset.py`, `model.py`, and
`pytorch_lightning.py`. Edit them directly here while iterating.

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import lightning.pytorch as pl
from torch import nn
from torch.utils.data import Dataset, DataLoader
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from PIL import Image

In [ ]:
# --- data_splitter.py ---

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


def parse_clip_frame_token(path: Path, class_name: str) -> tuple[str, float, str]:
    token = path.stem.rsplit("_", 1)[-1]
    if token.isdigit():
        number = int(token)
        clip_id = f"{class_name}|num_{number // 10000}"
        frame_idx = float(number % 10000)
        return clip_id, frame_idx, "numeric"

    match = re.fullmatch(r"([A-Za-z]+)(\d+)", token)
    if match:
        prefix, number = match.groups()
        clip_id = f"{class_name}|{prefix}{number}"
        return clip_id, float("nan"), "alpha_num"

    return f"{class_name}|{token}", float("nan"), "other"


def length_bucket(num_frames: int) -> str:
    if num_frames <= 2:
        return "1-2"
    if num_frames <= 5:
        return "3-5"
    if num_frames <= 10:
        return "6-10"
    if num_frames <= 20:
        return "11-20"
    if num_frames <= 50:
        return "21-50"
    return "51+"


def resolution_bucket(width: int, height: int) -> str:
    area = width * height
    if area <= 100_000:
        return "tiny"
    if area <= 250_000:
        return "small"
    if area <= 600_000:
        return "medium"
    return "large"


def sample_or_pad_indices(num_frames: int, sequence_len: int) -> np.ndarray:
    if num_frames <= 0:
        return np.zeros(sequence_len, dtype=int)
    if num_frames >= sequence_len:
        return np.linspace(0, num_frames - 1, sequence_len, dtype=int)
    indices = np.arange(num_frames, dtype=int)
    pad = np.full(sequence_len - num_frames, num_frames - 1, dtype=int)
    return np.concatenate([indices, pad])


def _collect_frame_rows(data_dir: Path) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for class_dir in sorted([path for path in data_dir.iterdir() if path.is_dir()]):
        class_name = class_dir.name
        for image_path in sorted(class_dir.iterdir()):
            if image_path.suffix.lower() not in IMAGE_EXTS:
                continue
            clip_id, frame_idx, token_type = parse_clip_frame_token(image_path, class_name)
            rows.append(
                {
                    "class": class_name,
                    "file": image_path.name,
                    "clip_id": clip_id,
                    "frame_idx": frame_idx,
                    "token_type": token_type,
                }
            )
    if not rows:
        raise FileNotFoundError(f"No image files found under {data_dir}")
    return pd.DataFrame(rows)


def _image_size(path: Path) -> tuple[int, int]:
    with Image.open(path) as image:
        return image.width, image.height


def build_clip_meta(df_frames: pd.DataFrame, data_dir: Path) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for (class_name, clip_id), group in df_frames.groupby(["class", "clip_id"]):
        group = group.sort_values(["frame_idx", "file"], na_position="last")
        first_file = group.iloc[0]["file"]
        image_path = data_dir / class_name / first_file
        width, height = _image_size(image_path)
        num_frames = len(group)
        rows.append(
            {
                "class": class_name,
                "clip_id": clip_id,
                "n_frames": num_frames,
                "width": width,
                "height": height,
                "length_bucket": length_bucket(num_frames),
                "resolution_bucket": resolution_bucket(width, height),
                "bucket_key": f"{length_bucket(num_frames)}|{resolution_bucket(width, height)}",
            }
        )
    return pd.DataFrame(rows)


def assign_split_frames(
    clip_meta: pd.DataFrame,
    seed: int,
    train_frac: float,
    val_frac: float,
    test_frac: float,
) -> pd.DataFrame:
    if abs((train_frac + val_frac + test_frac) - 1.0) > 1e-9:
        raise ValueError("train_frac + val_frac + test_frac must equal 1.0")

    rng = np.random.default_rng(seed)
    parts: list[pd.DataFrame] = []

    for class_name, group in clip_meta.groupby("class"):
        bucket_lists: list[list[str]] = []
        for _, bucket_group in group.groupby("bucket_key"):
            clip_ids = bucket_group["clip_id"].to_numpy().copy()
            rng.shuffle(clip_ids)
            bucket_lists.append(list(clip_ids))

        ordered_clip_ids: list[str] = []
        while any(bucket_lists):
            for bucket_ids in bucket_lists:
                if bucket_ids:
                    ordered_clip_ids.append(bucket_ids.pop())

        total = len(ordered_clip_ids)
        n_train = int(round(train_frac * total))
        n_val = int(round(val_frac * total))
        if total >= 3:
            n_train = max(1, min(n_train, total - 2))
            n_val = max(1, min(n_val, total - n_train - 1))
        split = np.array(["test"] * total, dtype=object)
        split[:n_train] = "train"
        split[n_train : n_train + n_val] = "val"
        parts.append(pd.DataFrame({"class": class_name, "clip_id": ordered_clip_ids, "split": split}))

    split_clips = pd.concat(parts, ignore_index=True)
    return split_clips


def build_sequence_manifest(df_frames_split: pd.DataFrame, sequence_len: int, data_dir: Path) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    frame_cols = [f"f{i:02d}" for i in range(sequence_len)]

    for (class_name, clip_id, split), group in df_frames_split.groupby(["class", "clip_id", "split"]):
        ordered = group.sort_values(["frame_idx", "file"], na_position="last")
        rel_paths = [f"data/{class_name}/{filename}" for filename in ordered["file"].tolist()]
        chosen = sample_or_pad_indices(len(rel_paths), sequence_len)
        first_row = ordered.iloc[0]
        row = {
            "class": class_name,
            "clip_id": clip_id,
            "split": split,
            "n_frames_orig": len(rel_paths),
            "frame_idx_min": ordered["frame_idx"].min(skipna=True),
            "frame_idx_max": ordered["frame_idx"].max(skipna=True),
            "token_type": first_row.get("token_type", "unknown"),
        }
        for idx, frame_col in enumerate(frame_cols):
            row[frame_col] = rel_paths[chosen[idx]]
        rows.append(row)

    sequence_manifest = pd.DataFrame(rows)
    label_map = pd.DataFrame({"class": sorted(sequence_manifest["class"].unique())})
    label_map["label_id"] = np.arange(len(label_map), dtype=int)

    sequence_manifest = sequence_manifest.merge(label_map, on="class", how="left")
    return sequence_manifest, label_map


def build_artifacts(
    data_dir: str | Path,
    artifacts_dir: str | Path,
    sequence_len: int,
    seed: int,
    train_frac: float,
    val_frac: float,
    test_frac: float,
) -> dict[str, Path]:
    data_dir = Path(data_dir).resolve()
    artifacts_dir = Path(artifacts_dir).resolve()
    artifacts_dir.mkdir(parents=True, exist_ok=True)

    df_frames = _collect_frame_rows(data_dir)
    clip_meta = build_clip_meta(df_frames, data_dir)
    split_clips = assign_split_frames(clip_meta, seed, train_frac, val_frac, test_frac)
    df_frames_split = df_frames.merge(split_clips, on=["class", "clip_id"], how="left")
    clip_meta_split = clip_meta.merge(split_clips, on=["class", "clip_id"], how="left")
    sequence_manifest, label_map = build_sequence_manifest(df_frames_split, sequence_len, data_dir)

    if sequence_manifest["label_id"].isna().any():
        raise RuntimeError("Label mapping failed while building the sequence manifest")

    split_clips_path = artifacts_dir / "clip_splits.csv"
    frame_manifest_path = artifacts_dir / "frame_manifest_with_split.csv"
    clip_meta_path = artifacts_dir / "clip_meta_with_split.csv"
    sequence_manifest_path = artifacts_dir / f"sequence_manifest_len{sequence_len}.csv"
    label_map_path = artifacts_dir / "label_map.csv"

    split_clips.to_csv(split_clips_path, index=False)
    df_frames_split.to_csv(frame_manifest_path, index=False)
    clip_meta_split.to_csv(clip_meta_path, index=False)
    sequence_manifest.to_csv(sequence_manifest_path, index=False)
    label_map.to_csv(label_map_path, index=False)

    return {
        "split_clips": split_clips_path,
        "frame_manifest": frame_manifest_path,
        "clip_meta": clip_meta_path,
        "sequence_manifest": sequence_manifest_path,
        "label_map": label_map_path,
    }

In [ ]:
# --- dataset.py ---


class WorkoutSequenceDataset(Dataset):
    def __init__(
        self,
        manifest_path: str | Path,
        data_root: str | Path,
        split: str | None = None,
        image_size: int | tuple[int, int] = 128,
        normalize: bool = True,
    ) -> None:
        self.manifest_path = Path(manifest_path)
        self.data_root = Path(data_root)
        self.image_size = image_size
        self.normalize = normalize

        df = pd.read_csv(self.manifest_path)
        if split is not None:
            df = df[df["split"] == split].reset_index(drop=True)
        if df.empty:
            raise ValueError(f"No rows found in {self.manifest_path} for split={split!r}")

        self.df = df
        self.frame_cols = sorted([column for column in self.df.columns if re.fullmatch(r"f\d\d", column)])
        if not self.frame_cols:
            raise ValueError(f"Manifest {self.manifest_path} does not contain fixed-length frame columns")

        self.mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(3, 1, 1)

    def __len__(self) -> int:
        return len(self.df)

    def _resize(self, image: Image.Image) -> Image.Image:
        if isinstance(self.image_size, int):
            size = (self.image_size, self.image_size)
        else:
            size = tuple(self.image_size)
        return image.resize(size, Image.BILINEAR)

    def _load_image(self, relative_path: str) -> torch.Tensor:
        path = self.data_root / relative_path
        with Image.open(path) as image:
            image = self._resize(image.convert("RGB"))
            array = np.asarray(image, dtype=np.float32) / 255.0
        tensor = torch.from_numpy(array).permute(2, 0, 1)
        if self.normalize:
            tensor = (tensor - self.mean) / self.std
        return tensor

    def __getitem__(self, index: int) -> dict[str, Any]:
        row = self.df.iloc[index]
        frames = [self._load_image(row[column]) for column in self.frame_cols]
        frames_tensor = torch.stack(frames, dim=0)
        label = torch.tensor(int(row["label_id"]), dtype=torch.long)
        return {
            "frames": frames_tensor,
            "label": label,
            "clip_id": str(row["clip_id"]),
            "class_name": str(row["class"]),
            "split": str(row["split"]),
        }

In [ ]:
# --- model.py ---


class FrameEncoder(nn.Module):
    def __init__(
        self,
        in_channels: int = 3,
        hidden_dims: Iterable[int] = (32, 64, 128),
        embedding_dim: int = 128,
        dropout: float = 0.2,
    ) -> None:
        super().__init__()
        blocks: list[nn.Module] = []
        current_channels = in_channels
        for hidden_dim in hidden_dims:
            blocks.extend(
                [
                    nn.Conv2d(current_channels, hidden_dim, kernel_size=3, padding=1, bias=False),
                    nn.BatchNorm2d(hidden_dim),
                    nn.ReLU(inplace=True),
                    nn.MaxPool2d(kernel_size=2),
                ]
            )
            current_channels = hidden_dim
        self.backbone = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(current_channels, embedding_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.backbone(x)
        x = self.pool(x)
        return self.projection(x)


class SequenceClassifier(nn.Module):
    def __init__(
        self,
        num_classes: int,
        in_channels: int = 3,
        hidden_dims: Iterable[int] = (32, 64, 128),
        embedding_dim: int = 128,
        dropout: float = 0.2,
        temporal_pooling: str = "mean",
    ) -> None:
        super().__init__()
        self.temporal_pooling = temporal_pooling.lower()
        self.frame_encoder = FrameEncoder(
            in_channels=in_channels,
            hidden_dims=hidden_dims,
            embedding_dim=embedding_dim,
            dropout=dropout,
        )

        if self.temporal_pooling == "lstm":
            hidden_size = max(16, embedding_dim // 2)
            self.temporal = nn.LSTM(
                input_size=embedding_dim,
                hidden_size=hidden_size,
                num_layers=1,
                batch_first=True,
                bidirectional=True,
            )
            feature_dim = hidden_size * 2
        else:
            self.temporal = None
            feature_dim = embedding_dim

        self.classifier = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Dropout(dropout),
            nn.Linear(feature_dim, num_classes),
        )

    def _aggregate_temporal(self, embeddings: torch.Tensor) -> torch.Tensor:
        if self.temporal_pooling == "mean":
            return embeddings.mean(dim=1)
        if self.temporal_pooling == "max":
            return embeddings.max(dim=1).values
        if self.temporal_pooling == "lstm":
            _, (hidden_state, _) = self.temporal(embeddings)
            forward_last = hidden_state[-2]
            backward_last = hidden_state[-1]
            return torch.cat([forward_last, backward_last], dim=1)
        raise ValueError(f"Unknown temporal_pooling={self.temporal_pooling!r}")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, sequence_length = x.shape[:2]
        frames = x.reshape(batch_size * sequence_length, *x.shape[2:])
        frame_embeddings = self.frame_encoder(frames).reshape(batch_size, sequence_length, -1)
        sequence_embedding = self._aggregate_temporal(frame_embeddings)
        return self.classifier(sequence_embedding)

In [ ]:
# --- pytorch_lightning.py ---


class WorkoutLightningModule(pl.LightningModule):
    def __init__(self, model: nn.Module, lr: float = 1e-3, weight_decay: float = 0.0) -> None:
        super().__init__()
        self.save_hyperparameters(ignore=["model"])
        self.model = model
        self.lr = lr
        self.weight_decay = weight_decay

    def forward(self, frames: torch.Tensor) -> torch.Tensor:
        return self.model(frames)

    def _step(self, batch: dict[str, torch.Tensor], stage: str) -> torch.Tensor:
        frames = batch["frames"]
        labels = batch["label"]
        logits = self(frames)
        loss = F.cross_entropy(logits, labels)
        predictions = logits.argmax(dim=1)
        accuracy = (predictions == labels).float().mean()
        self.log(f"{stage}_loss", loss, prog_bar=(stage != "train"), on_step=(stage == "train"), on_epoch=True)
        self.log(f"{stage}_acc", accuracy, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def training_step(self, batch: dict[str, torch.Tensor], batch_idx: int) -> torch.Tensor:
        return self._step(batch, "train")

    def validation_step(self, batch: dict[str, torch.Tensor], batch_idx: int) -> torch.Tensor:
        return self._step(batch, "val")

    def test_step(self, batch: dict[str, torch.Tensor], batch_idx: int) -> torch.Tensor:
        return self._step(batch, "test")

    def predict_step(self, batch: dict[str, torch.Tensor], batch_idx: int, dataloader_idx: int = 0):
        frames = batch["frames"]
        labels = batch["label"]
        logits = self(frames)
        probabilities = torch.softmax(logits, dim=1)
        confidence, predictions = probabilities.max(dim=1)
        return {
            "clip_id": batch["clip_id"],
            "class_name": batch["class_name"],
            "label": labels.detach().cpu(),
            "prediction": predictions.detach().cpu(),
            "confidence": confidence.detach().cpu(),
        }

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)

In [ ]:
# --- configs/base.yaml, as a dict ---

CONFIG: dict[str, Any] = {
    "mode": {
        "name": "colab_training",
        "seed": 42,
        "build_artifacts": True,
    },
    "data": {
        "sequence_len": 16,
        "image_size": 128,
        "batch_size": 16,
        "num_workers": 2,
        "train_frac": 0.70,
        "val_frac": 0.15,
        "test_frac": 0.15,
    },
    "model": {
        "in_channels": 3,
        "hidden_dims": [32, 64, 128],
        "embedding_dim": 128,
        "dropout": 0.20,
        "temporal_pooling": "mean",
    },
    "training": {
        "lr": 0.001,
        "weight_decay": 0.0001,
        "max_epochs": 10,
        "accelerator": "auto",
        "devices": "auto",
        "precision": "32-true",
        "log_every_n_steps": 10,
        "patience": 4,
        "checkpoint_dir": "artifacts/checkpoints",
    },
}

In [ ]:
def ensure_artifacts(cfg: dict[str, Any], project_root: Path) -> dict[str, Path]:
    mode_cfg = cfg.get('mode', {})
    data_cfg = cfg.get('data', {})
    data_dir = project_root / 'data'
    artifacts_dir = project_root / 'artifacts'
    sequence_len = int(data_cfg.get('sequence_len', 16))
    sequence_manifest = artifacts_dir / f'sequence_manifest_len{sequence_len}.csv'

    if bool(mode_cfg.get('build_artifacts', True)) or not sequence_manifest.exists():
        return build_artifacts(
            data_dir=data_dir,
            artifacts_dir=artifacts_dir,
            sequence_len=sequence_len,
            seed=int(mode_cfg.get('seed', 42)),
            train_frac=float(data_cfg.get('train_frac', 0.70)),
            val_frac=float(data_cfg.get('val_frac', 0.15)),
            test_frac=float(data_cfg.get('test_frac', 0.15)),
        )

    return {
        'split_clips': artifacts_dir / 'clip_splits.csv',
        'frame_manifest': artifacts_dir / 'frame_manifest_with_split.csv',
        'clip_meta': artifacts_dir / 'clip_meta_with_split.csv',
        'sequence_manifest': sequence_manifest,
        'label_map': artifacts_dir / 'label_map.csv',
    }


def build_dataloaders(cfg: dict[str, Any], project_root: Path):
    data_cfg = cfg.get('data', {})
    artifacts = ensure_artifacts(cfg, project_root)

    data_root = project_root / 'data'
    manifest_path = artifacts['sequence_manifest']
    image_size = int(data_cfg.get('image_size', 128))
    batch_size = int(data_cfg.get('batch_size', 16))
    num_workers = int(data_cfg.get('num_workers', 2))
    pin_memory = torch.cuda.is_available()

    train_dataset = WorkoutSequenceDataset(manifest_path, data_root, split='train', image_size=image_size)
    val_dataset = WorkoutSequenceDataset(manifest_path, data_root, split='val', image_size=image_size)
    test_dataset = WorkoutSequenceDataset(manifest_path, data_root, split='test', image_size=image_size)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=bool(num_workers),
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=bool(num_workers),
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=bool(num_workers),
    )

    label_map = pd.read_csv(artifacts['label_map'])
    num_classes = int(label_map['label_id'].nunique())
    return train_loader, val_loader, test_loader, num_classes, artifacts


def build_model(cfg: dict[str, Any], num_classes: int) -> SequenceClassifier:
    model_cfg = cfg.get('model', {})
    return SequenceClassifier(
        num_classes=num_classes,
        in_channels=int(model_cfg.get('in_channels', 3)),
        hidden_dims=tuple(model_cfg.get('hidden_dims', [32, 64, 128])),
        embedding_dim=int(model_cfg.get('embedding_dim', 128)),
        dropout=float(model_cfg.get('dropout', 0.20)),
        temporal_pooling=str(model_cfg.get('temporal_pooling', 'mean')),
    )


def build_lightning_module(cfg: dict[str, Any], num_classes: int) -> WorkoutLightningModule:
    training_cfg = cfg.get('training', {})
    model = build_model(cfg, num_classes=num_classes)
    return WorkoutLightningModule(
        model=model,
        lr=float(training_cfg.get('lr', 1e-3)),
        weight_decay=float(training_cfg.get('weight_decay', 0.0)),
    )


def build_trainer(cfg: dict[str, Any], project_root: Path) -> pl.Trainer:
    training_cfg = cfg.get('training', {})
    checkpoint_dir = project_root / training_cfg.get('checkpoint_dir', 'artifacts/checkpoints')
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    monitor = str(training_cfg.get('monitor', 'val_acc'))
    mode = str(training_cfg.get('monitor_mode', 'max'))
    callbacks = [
        ModelCheckpoint(
            dirpath=checkpoint_dir,
            filename='epoch{epoch:02d}-{val_acc:.3f}',
            monitor=monitor,
            mode=mode,
            save_top_k=1,
        ),
        EarlyStopping(
            monitor=monitor,
            mode=mode,
            patience=int(training_cfg.get('patience', 4)),
        ),
        LearningRateMonitor(logging_interval='epoch'),
    ]

    precision = training_cfg.get('precision', '32-true')
    if precision == 'auto':
        precision = '16-mixed' if torch.cuda.is_available() else '32-true'

    return pl.Trainer(
        max_epochs=int(training_cfg.get('max_epochs', 10)),
        accelerator=str(training_cfg.get('accelerator', 'auto')),
        devices=training_cfg.get('devices', 'auto'),
        precision=precision,
        log_every_n_steps=int(training_cfg.get('log_every_n_steps', 10)),
        default_root_dir=str(project_root / 'artifacts'),
        callbacks=callbacks,
    )


def _flatten_prediction_batches(prediction_batches: list[dict[str, Any]]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for batch in prediction_batches:
        clip_ids = list(batch['clip_id'])
        class_names = list(batch['class_name'])
        labels = batch['label'].tolist()
        predictions = batch['prediction'].tolist()
        confidences = batch['confidence'].tolist()
        for clip_id, class_name, label, prediction, confidence in zip(
            clip_ids, class_names, labels, predictions, confidences
        ):
            rows.append(
                {
                    'clip_id': clip_id,
                    'class_name': class_name,
                    'label': int(label),
                    'prediction': int(prediction),
                    'confidence': float(confidence),
                    'correct': int(label) == int(prediction),
                }
            )
    return pd.DataFrame(rows)


def run_training(cfg: dict[str, Any], project_root: Path) -> dict[str, Any]:
    seed = int(cfg.get('mode', {}).get('seed', 42))
    pl.seed_everything(seed, workers=True)

    train_loader, val_loader, test_loader, num_classes, artifacts = build_dataloaders(cfg, project_root)
    lit_module = build_lightning_module(cfg, num_classes=num_classes)
    trainer = build_trainer(cfg, project_root)

    trainer.fit(lit_module, train_loader, val_loader)
    test_results = trainer.test(lit_module, dataloaders=test_loader, verbose=False)
    prediction_batches = trainer.predict(lit_module, dataloaders=test_loader)

    prediction_frame = _flatten_prediction_batches(prediction_batches)
    prediction_path = artifacts['sequence_manifest'].parent / 'predictions.csv'
    prediction_frame.to_csv(prediction_path, index=False)

    metrics = {
        'project_root': str(project_root),
        'best_model_path': trainer.checkpoint_callback.best_model_path if trainer.checkpoint_callback else '',
        'test_results': test_results,
        'artifacts': {name: str(path) for name, path in artifacts.items()},
        'predictions_path': str(prediction_path),
    }

    metrics_path = artifacts['sequence_manifest'].parent / 'training_summary.json'
    with open(metrics_path, 'w', encoding='utf-8') as handle:
        json.dump(metrics, handle, indent=2)

    print(f'Saved training summary to {metrics_path}')
    print(f'Saved predictions to {prediction_path}')
    return metrics


results = run_training(CONFIG, PROJECT_ROOT)
results

In [ ]:
import json

summary_path = PROJECT_ROOT / 'artifacts' / 'training_summary.json'
with open(summary_path, 'r', encoding='utf-8') as handle:
    summary = json.load(handle)

summary